## OpenAiSdk with OpenRouter for the Week 2 day 2


1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, set_tracing_disabled
from typing import Dict
import os
import asyncio
import json
import requests

# Note: This notebook is configured to work with OpenRouter by mapping
# OPENROUTER_API_KEY / OPENROUTER_API_BASE to OpenAI-style env vars.
# Pushover is used for sending notifications/emails (no SendGrid required).


In [2]:
load_dotenv()

# Configure OpenRouter (map to OpenAI-compatible env vars if provided)
# You do NOT need to put the model name in your .env — the notebook sets a default.
if os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
if os.getenv("OPENROUTER_API_BASE"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("OPENROUTER_API_BASE")

# Set tracing to disabled when using OpenRouter to avoid openai tracing errors
if os.getenv("OPENROUTER_API_KEY"):
    set_tracing_disabled(True)

# Default model (set here; no need to add the model to .env)
MODEL = "openai/gpt-oss-20b:free"

# Pushover configuration (used for sending notifications/emails)
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

print(f"Using model: {MODEL}")


Pushover user found and starts with u
Pushover token found and starts with a
Using model: openai/gpt-oss-20b:free


## Step 1: Agent workflow

In [3]:

instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails. Only answer with the email, no explanation."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response. Only answer with the email, no explanation"

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails. Only answer with the email, no explanation."


In [4]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=MODEL
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=MODEL
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=MODEL
)


In [5]:
message = "Write a cold sales email"

result = await Runner.run(sales_agent1, message)
print(result.final_output)

Subject: Accelerate Your SOC 2 Readiness with AI‑Powered Compliance

Hi [First Name],

I hope you’re doing well. I’m reaching out because I’ve seen many security teams struggle to keep SOC 2 documentation up‑to‑date while juggling day‑to‑day operations. ComplAI is a SaaS platform that automates the entire compliance lifecycle— from evidence collection and policy management to audit‑ready reporting—using advanced AI.

Key benefits for teams like yours:

* **30 % faster evidence gathering** – AI scans your tools and pulls the exact logs and screenshots needed for each control.
* **Real‑time risk scoring** – Continuous monitoring flags gaps before they become audit findings.
* **One‑click audit reports** – Generate complete, audit‑ready documentation in minutes, not weeks.
* **Seamless integration** – Works with Jira, Confluence, GitHub, and your existing security stack.

I’d love to show you how ComplAI can reduce your audit preparation time by up to 50 % while keeping your team focused 

In [6]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Accelerate Your SOC 2 Readiness with AI‑Powered Compliance

Hi [First Name],

I hope you’re doing well. I’m reaching out because I’ve seen many security teams struggle to keep SOC 2 controls up‑to‑date while juggling day‑to‑day operations. ComplAI is a SaaS platform that automates the entire compliance lifecycle—continuous monitoring, evidence collection, and audit‑ready reporting—using advanced AI.

Key benefits for teams like yours:

* **Real‑time control coverage** – AI maps your processes to SOC 2 requirements and flags gaps instantly.  
* **Automated evidence generation** – No more manual spreadsheets; the platform pulls logs, policies, and screenshots directly from your tools.  
* **Audit‑ready dashboards** – One‑click export of all evidence, ready for auditors, saving weeks of prep time.  
* **Scalable for growth** – Works across cloud, on‑prem, and hybrid environments without extra configuration.

I’d love to show you how ComplAI can reduce your audit preparation time 

In [7]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only no explanations. If there are any explanations or things like: here is a cold sales..., delete them.",
    model=MODEL
)


In [8]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Subject: Accelerate Your SOC 2 Readiness with AI‑Powered Compliance

Hi [First Name],

I hope you’re doing well. I’m reaching out because I’ve seen many security teams struggle to keep SOC 2 controls up to date while juggling day‑to‑day operations. ComplAI is a SaaS platform that automates the entire compliance lifecycle—continuous monitoring, evidence collection, and audit‑ready reporting—using AI to reduce manual effort by up to 70 %.

Key benefits for teams like yours:

* **Real‑time control monitoring** – Detect gaps before they become audit findings.  
* **Automated evidence generation** – One click produces the documentation auditors need.  
* **Audit‑ready dashboards** – Share progress with stakeholders without digging through spreadsheets.  
* **Scalable for growth** – Add new services or teams without re‑engineering your compliance process.

I’d love to show you how ComplAI can cut your audit preparation time from weeks to days. Are you available for a 15‑min

Now go and check out the **trace**:

`Openrouter`doesnt have any **traces**

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [9]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=MODEL,
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=MODEL,
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=MODEL,
)


In [10]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails. Only answer with the email, no explanation.', prompt=None, handoffs=[], model='openai/gpt-oss-20b:free', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [11]:
import os
import requests
from typing import Dict

PUSHOVER_API_URL = "https://api.pushover.net/1/messages.json"

@function_tool
def send_email(body: str) -> Dict[str, str]:
    """
    Send a notification through the Pushover API.
    """
    payload = {
        "token": os.environ["PUSHOVER_TOKEN"],
        "user": os.environ["PUSHOVER_USER"],
        "title": "Sales Notification",
        "message": body,
    }

    try:
        response = requests.post(
            PUSHOVER_API_URL,
            data=payload,
            timeout=10,
        )
        response.raise_for_status()
        return {"status": "success"}
    except requests.RequestException as error:
        return {
            "status": "error",
            "message": str(error),
        }



### This has automatically been converted into a tool, with the boilerplate json created

In [12]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send a notification through the Pushover API.', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78d88b7065c0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### And you can also convert an Agent into a tool

In [13]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78d8885800e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [14]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78d88904fc40>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x78d888580220>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='

## And now it's time for our Sales Manager - our planning agent

In [15]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message) 
    print(result)

    # Send Pushover notification
    if pushover_token and pushover_user and pushover_url:
        import requests
        data = {
            "token": pushover_token,
            "user": pushover_user,
            "message": "Sales email generated successfully!"
        }
        response = requests.post(pushover_url, data=data)
        print(f"Pushover notification sent: {response.status_code}")
    else:
        print("Pushover credentials missing. Notification not sent.")

# Send push notification with Pushover
if pushover_token and pushover_user and pushover_url:
    import requests
    data = {
        "token": pushover_token,
        "user": pushover_user,
        "message": "Sales email generated successfully",
        "title": "ComplAI Sales"
    }
    requests.post(pushover_url, data=data)

    print(result)


RunResult:
- Last agent: Agent(name="Sales Manager", ...)
- Final output (str):

- 0 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)
Pushover notification sent: 200
RunResult:
- Last agent: Agent(name="Sales Manager", ...)
- Final output (str):

- 0 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [16]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=MODEL)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=MODEL)
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")



In [17]:
import os
import requests
from typing import Dict

PUSHOVER_API_URL = "https://api.pushover.net/1/messages.json"

@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """
    Send a notification with an HTML-formatted message through the Pushover API.
    """
    payload = {
        "token": os.environ["PUSHOVER_TOKEN"],
        "user": os.environ["PUSHOVER_USER"],
        "title": subject,
        "message": html_body,
        "html": 1,
    }

    try:
        response = requests.post(
            PUSHOVER_API_URL,
            data=payload,
            timeout=10,
        )
        response.raise_for_status()
        return {"status": "success"}
    except requests.RequestException as error:
        return {
            "status": "error",
            "message": str(error),
        }


In [ ]:
# Let's look at it
tools = [subject_tool, html_tool, send_html_email]

In [ ]:
tools

In [ ]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model=MODEL,
    handoff_description="Convert an email to HTML and send it")



### Now we have 3 tools and 1 handoff

In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

In [ ]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=MODEL)

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)
